# **DME Traninning Data**

## **Import Flagged Fraud Claims**

In [1]:
import os
import pandas as pd
df_fraudclaims = pd.read_excel(r'U:\Zeming\AAA-Project\DME Fraud Claims Detection\Training Data\Collected Fraud Claims\Fraud_Claims.xlsx',sheet_name='Final List')
df_fraudclaims['cur_clm_uniq_id'] = df_fraudclaims['cur_clm_uniq_id'].astype('str')

## **Insert Raw DME Claim Data into Temp_Zeming**

In [12]:
with open(r'C:\Users\zliu\OneDrive - PBACO\Desktop\Traning Doc\Credentials\Credentials.txt', 'r') as f:
     credentials = [line.strip() for line in f.read().splitlines()]

In [4]:
#connect to database
import pyodbc
from urllib.parse import quote_plus
from sqlalchemy import create_engine

server = credentials[0]
database = credentials[1]
username = credentials[2]
password = credentials[3]
engine = create_engine(f"mssql+pyodbc://{username}:{password}@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server")

In [ ]:
#Instert Data into Temp DB
Insert_Query ='''

--All DME Claims Since 2020(look at claims back to 2 years before oldest 2021 flag claim, only for patients since 2021)
DROP TABLE IF EXISTS temp_zeming.dbo.dme
SELECT A.*
INTO temp_zeming.dbo.dme
FROM [_Internal_Reporting].[dbo].[CCLF6_PartB_DME_Claim_Line_Level_Amt] A
JOIN (SELECT DISTINCT BENEFICIARY_ID FROM [_Internal_Reporting].[dbo].[Assignment_NPI_Level_CurrentRules_Union] WHERE YEAR IN (2021,2022,2023,2024,2025) AND PERIOD IN (1,2,3,4)) B 
	ON A.Beneficiary_ID=B.Beneficiary_ID 
WHERE clm_thru_dt BETWEEN '2019-01-01' AND '2025-12-31'

CREATE NONCLUSTERED INDEX IX_dme_ordrg_npi_date
ON temp_zeming.dbo.dme (ordrg_prvdr_npi_num, clm_thru_dt)
INCLUDE (cur_clm_uniq_id, clm_line_hcpcs_cd, clm_line_alowd_chrg_amt, clm_line_cvrd_pd_amt);

--Claim Header Level Info
DROP TABLE IF Exists temp_zeming.dbo.dme_header
SELECT cur_clm_uniq_id, MAX(CLM_FROM_DT) AS CLM_FROM_DT, MAX(clm_thru_dt) AS CLM_THRU_DT
, MAX(ordrg_prvdr_npi_num) AS  ordrg_prvdr_npi_num
, MAX(pay_to_prvdr_npi_num) AS  pay_to_prvdr_npi_num
, MAX(Beneficiary_ID) AS Beneficiary_ID
INTO temp_zeming.dbo.dme_header
FROM temp_zeming.dbo.dme
GROUP BY cur_clm_uniq_id

CREATE NONCLUSTERED INDEX IX_header_ordrg_npi_date
ON temp_zeming.dbo.dme_header (ordrg_prvdr_npi_num, clm_thru_dt)
INCLUDE (cur_clm_uniq_id, Beneficiary_ID);
'''

with engine.begin() as conn:
    conn.exec_driver_sql(Insert_Query)

## **Feature Tables Claim Level**

In [6]:
#pull data from sql server
query_dme = f'''
SELECT * FROM temp_zeming.dbo.dme WHERE clm_thru_dt BETWEEN '2021-01-01' AND '2025-12-31'
'''
df_features_claim=pd.read_sql_query(query_dme,engine)

query_high_risk_cpt = f'''
SELECT DISTINCT CPT FROM [temp_Zeming].[dbo].[DMEAutomation_CPT]
'''
df_high_risk_cpt = pd.read_sql_query(query_high_risk_cpt,engine)

ERROR! Session/line number was not unique in database. History logging moved to new session 1170


In [7]:
df_features_claim['clm_from_dt'] = pd.to_datetime(df_features_claim['clm_from_dt'], errors='coerce')
df_features_claim['clm_thru_dt'] = pd.to_datetime(df_features_claim['clm_thru_dt'], errors='coerce')

df_features_claim = (
    df_features_claim.groupby('cur_clm_uniq_id').agg(
        clm_chrg_amt=('clm_line_alowd_chrg_amt', 'sum'),
        clm_paid_amt=('clm_line_cvrd_pd_amt','sum'),
        clm_lines_ct=('clm_line_num', 'count'),
        clm_all_cpts = ('clm_line_hcpcs_cd', lambda x: list(x.dropna().unique())),
        clm_uniq_cpt_ct=('clm_line_hcpcs_cd', 'nunique'),
        clm_from_dt=('clm_from_dt', 'min'),
        clm_thru_dt=('clm_thru_dt', 'max'),
        clm_status=('clm_adjsmt_type_cd','max'),
        clm_beneficiary_id=('Beneficiary_ID','first'),
        clm_billnpi=('pay_to_prvdr_npi_num','first'),
        clm_ordenpi=('ordrg_prvdr_npi_num','first')
        
    )
    .reset_index()
)

#date features
df_features_claim['clm_from_month'] = df_features_claim['clm_from_dt'].dt.month
df_features_claim['clm_from_day'] = df_features_claim['clm_from_dt'].dt.weekday
df_features_claim['clm_duration_days'] = (df_features_claim['clm_thru_dt'] - df_features_claim['clm_from_dt']).dt.days+1

#ori claim flag
df_features_claim['original_clm_flag'] = df_features_claim['clm_status'].map({0:0}).fillna(1).astype(int)

#fraud flag
df_features_claim = df_features_claim.merge(df_fraudclaims, how='left', left_on='cur_clm_uniq_id', right_on='cur_clm_uniq_id',suffixes=('','_right'))
right_cols = [c for c in df_features_claim.columns if c.endswith('_right')]
df_features_claim.drop(columns=right_cols, inplace=True)

#high risk cpt flag
high_risk_set = set(df_high_risk_cpt['CPT'])
df_features_claim['risk_cpt_flag'] = df_features_claim['clm_all_cpts'].apply(
    lambda lst: int(len(set(lst) & high_risk_set) > 0)
)

#drop Status
df_features_claim.drop(columns=['clm_status'],inplace=True)

In [8]:
print(df_features_claim.columns)
df_features_claim.to_csv(r'U:\Zeming\AAA-Project\DME Fraud Claims Detection\Training Data\features_claim.csv')

Index(['cur_clm_uniq_id', 'clm_chrg_amt', 'clm_paid_amt', 'clm_lines_ct',
       'clm_all_cpts', 'clm_uniq_cpt_ct', 'clm_from_dt', 'clm_thru_dt',
       'clm_beneficiary_id', 'clm_billnpi', 'clm_ordenpi', 'clm_from_month',
       'clm_from_day', 'clm_duration_days', 'original_clm_flag', 'fraud_flag',
       'risk_cpt_flag'],
      dtype='object')


## **Feature Tables Bene Level**

In [9]:
#Claim Line Counts in 30, 90, 365 days
#Distinct Claim Header Count for this patient in 30, 90, 365 days
#distinct HCPCS Count in 30, 90, 365 days
#Total Allowed Amount in 30, 90, 365 days
#Total Paid Amount in 30, 90, 365 days
#Total High Risk CPT Counts 30, 90, 365 days
#Number of Distinct Ordering Providers Seen in 30, 90, 365 days
#Number of Distinct Billing Providers Seen in 30, 90, 365 days

#Days_since_last_claim
#Number of Seen This Ordering Providers Seen in 30, 90, 365 days
#Number of Seen This Billing Providers Seen in 30, 90, 365 days
#Bene EVER Seen Ordering Providers in Last 5 Years

#Avg Allow Amount Per Claim
#Avg Paid Amount Per Claim
#Orderding / Billing

sql_build_tables_bene = f'''
--Look Back 30, 90, 365days
DROP TABLE IF EXISTS #BENE_FEATURES_1;
SELECT
    a.cur_clm_uniq_id,

--Cliam Line Counts
    COUNT( CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS bene_clmline_ct_30days,

    COUNT( CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS bene_clmline_ct_90days,

    COUNT( CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS bene_clmline_ct_365days,

--Claim Header Counts
    COUNT( DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS bene_clmheader_ct_30days,

    COUNT( DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS bene_clmheader_ct_90days,

    COUNT( DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS bene_clmheader_ct_365days,

	--distinct hcpcs
	COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.clm_line_hcpcs_cd
    END) AS bene_uniq_cpt_ct_30days,

    COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.clm_line_hcpcs_cd
    END) AS bene_uniq_cpt_ct_90days,

    COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.clm_line_hcpcs_cd
    END) AS bene_uniq_cpt_ct_365days,

	--total allowed amount
	SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.clm_line_alowd_chrg_amt
    END) AS bene_chrg_amt_30days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.clm_line_alowd_chrg_amt
    END) AS bene_chrg_amt_90days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.clm_line_alowd_chrg_amt
    END) AS bene_chrg_amt_365days,

	--total paid amount
	SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.clm_line_cvrd_pd_amt
    END) AS bene_paid_amt_30days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.clm_line_cvrd_pd_amt
    END) AS bene_paid_amt_90days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.clm_line_cvrd_pd_amt
    END) AS bene_paid_amt_365days,

	--high risk cpt count
	SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt) AND C.CPT IS NOT NULL THEN 1 
		ELSE 0
    END) AS bene_risk_cpt_ct_30days,

	SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt) AND C.CPT IS NOT NULL THEN 1 
		ELSE 0
    END) AS bene_risk_cpt_ct_90days,

	SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365,  a.clm_thru_dt) AND C.CPT IS NOT NULL THEN 1 
		ELSE 0
    END) AS bene_risk_cpt_ct_365days,

	--distinct ordering providers
	COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.ordrg_prvdr_npi_num
    END) AS bene_uniq_ordenpi_30days,

    COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.ordrg_prvdr_npi_num
    END) AS bene_uniq_ordenpi_90days,

    COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.ordrg_prvdr_npi_num
    END) AS bene_uniq_ordenpi_365days,

	--distinct billing providers
	COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.pay_to_prvdr_npi_num
    END) AS bene_uniq_billnpi_30days,

    COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.pay_to_prvdr_npi_num
    END) AS bene_uniq_billnpi_90days,

    COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.pay_to_prvdr_npi_num
    END) AS bene_uniq_billnpi_365days

INTO #BENE_FEATURES_1
FROM temp_zeming.dbo.dme_header a
JOIN temp_zeming.dbo.dme b
  ON a.Beneficiary_ID = b.Beneficiary_ID
  AND b.clm_thru_dt <= a.clm_thru_dt --claims before the anchor date
  AND b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)   -- maxmium 365 days look back
LEFT JOIN (SELECT DISTINCT CPT FROM [temp_Zeming].[dbo].[DMEAutomation_CPT]) c on b.clm_line_hcpcs_cd = c.CPT
WHERE a.clm_thru_dt >= '2021-01-01'
GROUP BY
    a.cur_clm_uniq_id



--days since last claim
DROP TABLE IF EXISTS #BENE_FEATURES_2;
SELECT A.cur_clm_uniq_id, MIN(DATEDIFF(DAY,B.clm_thru_dt,A.clm_thru_dt)) as bene_days_last_claim
INTO #BENE_FEATURES_2
FROM temp_zeming.dbo.dme_header a
LEFT JOIN temp_zeming.dbo.dme b
  ON a.Beneficiary_ID = b.Beneficiary_ID
 AND b.clm_thru_dt <= a.clm_thru_dt
 AND b.clm_thru_dt >= DATEADD(day,-365,a.clm_thru_dt)
 AND b.cur_clm_uniq_id <> a.cur_clm_uniq_id
WHERE a.clm_thru_dt >= '2021-01-01'
GROUP BY A.cur_clm_uniq_id

--Number see this provider in 30/90/365 days
DROP TABLE IF EXISTS #BENE_FEATURES_3;
SELECT 
    A.cur_clm_uniq_id,
    COUNT(DISTINCT CASE WHEN b.clm_thru_dt >= DATEADD(day, -30, a.clm_thru_dt) AND A.pay_to_prvdr_npi_num =B.pay_to_prvdr_npi_num
		THEN b.cur_clm_uniq_id END) AS bene_seen_billnpi_ct_30days,
	COUNT(DISTINCT CASE WHEN b.clm_thru_dt >= DATEADD(day, -90, a.clm_thru_dt) AND A.pay_to_prvdr_npi_num =B.pay_to_prvdr_npi_num
		THEN b.cur_clm_uniq_id END) AS bene_seen_billnpi_ct_90days,
	COUNT(DISTINCT CASE WHEN b.clm_thru_dt >= DATEADD(day,-365, a.clm_thru_dt) AND A.pay_to_prvdr_npi_num =B.pay_to_prvdr_npi_num
		THEN b.cur_clm_uniq_id END) AS bene_seen_billnpi_ct_365days,
	COUNT(DISTINCT CASE WHEN b.clm_thru_dt >= DATEADD(day,-30, a.clm_thru_dt) AND A.ordrg_prvdr_npi_num =B.ordrg_prvdr_npi_num
		THEN b.cur_clm_uniq_id END) AS bene_seen_ordnpi_ct_30days,
	COUNT(DISTINCT CASE WHEN b.clm_thru_dt >= DATEADD(day,-90, a.clm_thru_dt) AND A.ordrg_prvdr_npi_num =B.ordrg_prvdr_npi_num
		THEN b.cur_clm_uniq_id END) AS bene_seen_ordnpi_ct_90days,
	COUNT(DISTINCT CASE WHEN b.clm_thru_dt >= DATEADD(day,-365, a.clm_thru_dt) AND A.ordrg_prvdr_npi_num =B.ordrg_prvdr_npi_num
		THEN b.cur_clm_uniq_id END) AS bene_seen_ordnpi_ct_365days
INTO #BENE_FEATURES_3
FROM temp_zeming.dbo.dme_header a
LEFT JOIN temp_zeming.dbo.dme b
  ON a.Beneficiary_ID = b.Beneficiary_ID
 AND b.clm_thru_dt < a.clm_thru_dt
 AND b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
GROUP BY A.cur_clm_uniq_id;

--Patients Not See Ordering Providers In Part A and B in last 5 years
DROP TABLE IF EXISTS #BENE_FEATURES_4;
;WITH CTE2 AS --pts with operating and rendering provider
(
	SELECT DISTINCT Beneficiary_ID, oprtg_prvdr_npi_num AS Provider, clm_thru_dt
	FROM [_Internal_Reporting].[dbo].[CCLF_PartA_Claim_Line_Level_Amt] A
	WHERE clm_thru_dt BETWEEN '2016-01-01' AND '2025-12-31'
	UNION
	select DISTINCT Beneficiary_ID,RNDRG_PRVDR_NPI_NUM AS Provider, clm_thru_dt
	FROM [_Internal_Reporting].[dbo].[CCLF5_PartB_Claim_Line_Level_Amt]
	WHERE clm_thru_dt BETWEEN '2016-01-01' AND '2025-12-31'
) 

SELECT DISTINCT A.cur_clm_uniq_id 
INTO #BENE_FEATURES_4 --Bene Not Seen
FROM temp_zeming.dbo.dme_header A 
LEFT JOIN CTE2 B 
ON A.Beneficiary_ID=B.Beneficiary_ID AND A.ordrg_prvdr_npi_num=B.PROVIDER
AND B.clm_thru_dt BETWEEN DATEADD(YEAR,-5, A.clm_thru_dt) AND A.clm_thru_dt
WHERE B.Beneficiary_ID IS NULL
'''

final_sql_bene = f'''
SELECT 
    A.*, 
    B.*, 
    C.*, 
    CASE 
        WHEN D.cur_clm_uniq_id IS NULL THEN 1 
        ELSE 0 
    END AS bene_seen_ordnpi_5years
FROM #BENE_FEATURES_1 A
LEFT JOIN #BENE_FEATURES_2 B ON A.cur_clm_uniq_id = B.cur_clm_uniq_id
LEFT JOIN #BENE_FEATURES_3 C ON A.cur_clm_uniq_id = C.cur_clm_uniq_id
LEFT JOIN #BENE_FEATURES_4 D ON A.cur_clm_uniq_id = D.cur_clm_uniq_id;
'''

from sqlalchemy import text
with engine.begin() as conn:
    conn.exec_driver_sql(sql_build_tables_bene)
    df_features_bene = pd.read_sql_query(final_sql_bene, conn)

In [10]:
#Calculated Fields

#Average Charged Amt 30, 90 365 days
df_features_bene['bene_avg_chrg_amt_per_claim_30days'] = df_features_bene['bene_chrg_amt_30days']/df_features_bene['bene_clmheader_ct_30days']
df_features_bene['bene_avg_chrg_amt_per_claim_90days'] = df_features_bene['bene_chrg_amt_90days']/df_features_bene['bene_clmheader_ct_90days']
df_features_bene['bene_avg_chrg_amt_per_claim_365days'] = df_features_bene['bene_chrg_amt_365days']/df_features_bene['bene_clmheader_ct_365days']

#Average Paid Amt 30, 90 365 days
df_features_bene['bene_avg_paid_amt_per_claim_30days'] = df_features_bene['bene_paid_amt_30days']/df_features_bene['bene_clmheader_ct_30days']
df_features_bene['bene_avg_paid_amt_per_claim_90days'] = df_features_bene['bene_paid_amt_90days']/df_features_bene['bene_clmheader_ct_90days']
df_features_bene['bene_avg_paid_amt_per_claim_365days'] = df_features_bene['bene_paid_amt_365days']/df_features_bene['bene_clmheader_ct_365days']

#Bill Orde Provider Percentage
df_features_bene['bene_bill_Orde_%_30days'] = df_features_bene['bene_uniq_billnpi_30days']/df_features_bene['bene_uniq_ordenpi_30days']
df_features_bene['bene_bill_Orde_%_90days'] = df_features_bene['bene_uniq_billnpi_90days']/df_features_bene['bene_uniq_ordenpi_90days']
df_features_bene['bene_bill_Orde_%_365days'] = df_features_bene['bene_uniq_billnpi_365days']/df_features_bene['bene_uniq_ordenpi_365days']

In [11]:
#Exports
print(df_features_bene.columns)
df_features_bene.to_csv(r'U:\Zeming\AAA-Project\DME Fraud Claims Detection\Training Data\features_bene.csv')

Index(['cur_clm_uniq_id', 'bene_clmline_ct_30days', 'bene_clmline_ct_90days',
       'bene_clmline_ct_365days', 'bene_clmheader_ct_30days',
       'bene_clmheader_ct_90days', 'bene_clmheader_ct_365days',
       'bene_uniq_cpt_ct_30days', 'bene_uniq_cpt_ct_90days',
       'bene_uniq_cpt_ct_365days', 'bene_chrg_amt_30days',
       'bene_chrg_amt_90days', 'bene_chrg_amt_365days', 'bene_paid_amt_30days',
       'bene_paid_amt_90days', 'bene_paid_amt_365days',
       'bene_risk_cpt_ct_30days', 'bene_risk_cpt_ct_90days',
       'bene_risk_cpt_ct_365days', 'bene_uniq_ordenpi_30days',
       'bene_uniq_ordenpi_90days', 'bene_uniq_ordenpi_365days',
       'bene_uniq_billnpi_30days', 'bene_uniq_billnpi_90days',
       'bene_uniq_billnpi_365days', 'cur_clm_uniq_id', 'bene_days_last_claim',
       'cur_clm_uniq_id', 'bene_seen_billnpi_ct_30days',
       'bene_seen_billnpi_ct_90days', 'bene_seen_billnpi_ct_365days',
       'bene_seen_ordnpi_ct_30days', 'bene_seen_ordnpi_ct_90days',
       'bene_see

## **Feature Tables Billing NPI Level**

In [ ]:
##Billing NPI Level
sql_build_tables_billnpi = f'''
DROP TABLE IF EXISTS #DME_BillNPI_DAILY;
SELECT
    pay_to_prvdr_npi_num,
    clm_thru_dt,

    COUNT(*) AS clmline_cnt,
    COUNT(DISTINCT cur_clm_uniq_id) AS dist_claim_cnt,
    COUNT(DISTINCT Beneficiary_ID) AS dist_bene_cnt,
    COUNT(DISTINCT clm_line_hcpcs_cd) AS dist_cpt_cnt,

    SUM(clm_line_alowd_chrg_amt) AS allowed_amt,
    SUM(clm_line_cvrd_pd_amt) AS paid_amt,

    SUM(CASE WHEN c.CPT IS NOT NULL THEN 1 ELSE 0 END) AS risk_cpt_cnt
INTO #DME_BillNPI_DAILY
FROM temp_zeming.dbo.dme d
LEFT JOIN (
    SELECT DISTINCT CPT
    FROM temp_zeming.dbo.DMEAutomation_CPT
) c
  ON d.clm_line_hcpcs_cd = c.CPT
GROUP BY
    pay_to_prvdr_npi_num,
    clm_thru_dt;

DROP TABLE IF EXISTS #BillNPI_FEATURES_1;
SELECT
    a.cur_clm_uniq_id,

    -- claim / line
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.clmline_cnt END) AS billnpi_clmline_ct_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.clmline_cnt END) AS billnpi_clmline_ct_90days,
    SUM(b.clmline_cnt) AS billnpi_clmline_ct_365days,

    -- distinct claim
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.dist_claim_cnt END) AS billnpi_clmheader_ct_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.dist_claim_cnt END) AS billnpi_clmheader_ct_90days,
    SUM(b.dist_claim_cnt) AS billnpi_clmheader_ct_365days,

    -- distinct bene
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.dist_bene_cnt END) AS billnpi_uniq_bene_ct_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.dist_bene_cnt END) AS billnpi_uniq_bene_ct_90days,
    SUM(b.dist_bene_cnt) AS billnpi_uniq_bene_ct_365days,

    -- amounts
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.allowed_amt END) AS billnpi_chrg_amt_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.allowed_amt END) AS billnpi_chrg_amt_90days,
    SUM(b.allowed_amt) AS billnpi_chrg_amt_365days,

    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.paid_amt END) AS billnpi_paid_amt_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.paid_amt END) AS billnpi_paid_amt_90days,
    SUM(b.paid_amt) AS billnpi_paid_amt_365days,

    -- risk CPT
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.risk_cpt_cnt END) AS billnpi_risk_cpt_ct_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.risk_cpt_cnt END) AS billnpi_risk_cpt_ct_90days,
    SUM(b.risk_cpt_cnt) AS billnpi_risk_cpt_ct_365days

INTO #BillNPI_FEATURES_1
FROM temp_zeming.dbo.dme_header a
JOIN #DME_BillNPI_DAILY b
  ON a.pay_to_prvdr_npi_num = b.pay_to_prvdr_npi_num
 AND b.clm_thru_dt BETWEEN DATEADD(day,-365,a.clm_thru_dt)
                       AND a.clm_thru_dt
WHERE a.clm_thru_dt >= '2021-01-01'
GROUP BY
    a.cur_clm_uniq_id;

--New Patient Counts
;WITH Claim_With_New_Flag AS (
    SELECT
        a.cur_clm_uniq_id,
        a.Beneficiary_ID,
        a.pay_to_prvdr_npi_num,
        a.clm_thru_dt,

        CASE 
            WHEN NOT EXISTS (
                SELECT 1
                FROM temp_zeming.dbo.dme_header x
                WHERE x.Beneficiary_ID = a.Beneficiary_ID
                  AND x.pay_to_prvdr_npi_num = a.pay_to_prvdr_npi_num
                  AND x.clm_thru_dt <= a.clm_thru_dt
                  AND x.clm_thru_dt >= DATEADD(year, -2, a.clm_thru_dt)
				  AND x.cur_clm_uniq_id != a.cur_clm_uniq_id
            )
            THEN 1
            ELSE 0
        END AS is_new_patient_provider
    FROM temp_zeming.dbo.dme_header a
)

SELECT
    a.cur_clm_uniq_id,

    COUNT(DISTINCT CASE 
        WHEN b.is_new_patient_provider = 1
         AND b.clm_thru_dt >= DATEADD(day, -30, a.clm_thru_dt)
        THEN b.Beneficiary_ID
    END) AS billnpi_new_bene_ct_30days,

    COUNT(DISTINCT CASE 
        WHEN b.is_new_patient_provider = 1
         AND b.clm_thru_dt >= DATEADD(day, -90, a.clm_thru_dt)
        THEN b.Beneficiary_ID
    END) AS billnpi_new_bene_ct_90days,

    COUNT(DISTINCT CASE 
        WHEN b.is_new_patient_provider = 1
         AND b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.Beneficiary_ID
    END) AS billnpi_new_bene_ct_365days
into #BillNPI_FEATURES_2
FROM Claim_With_New_Flag a
LEFT JOIN Claim_With_New_Flag b
  ON a.pay_to_prvdr_npi_num = b.pay_to_prvdr_npi_num
 AND b.clm_thru_dt <= a.clm_thru_dt
 AND b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
 AND b.cur_clm_uniq_id != a.cur_clm_uniq_id
WHERE a.clm_thru_dt >= '2021-01-01'
GROUP BY
    a.cur_clm_uniq_id;


--Top1, Top5 CPT count, allowed amount, paid amount in last 365 days for this provider
WITH CPT_History AS (
    SELECT
        a.cur_clm_uniq_id AS anchor_clm_id,
        a.pay_to_prvdr_npi_num,
        b.clm_line_hcpcs_cd,
        COUNT(b.cur_clm_uniq_id) AS cpt_claim_count,
		SUM(B.clm_line_alowd_chrg_amt) AS cpt_allowed_amt,
		SUM(B. clm_line_cvrd_pd_amt) as cpt_coverd_amt
    FROM temp_zeming.dbo.dme_header a
    JOIN temp_zeming.dbo.dme b
      ON a.pay_to_prvdr_npi_num = b.pay_to_prvdr_npi_num
     AND b.clm_thru_dt <= a.clm_thru_dt
     AND b.clm_thru_dt >= DATEADD(year, -1, a.clm_thru_dt)  -- lookback
    GROUP BY
        a.cur_clm_uniq_id,
        a.pay_to_prvdr_npi_num,
        b.clm_line_hcpcs_cd
)

, CPT_Ranked AS (
    SELECT
        anchor_clm_id AS cur_clm_uniq_id,
        pay_to_prvdr_npi_num,
        clm_line_hcpcs_cd,
        cpt_claim_count,
		cpt_allowed_amt,
		cpt_coverd_amt,
        ROW_NUMBER() OVER (
            PARTITION BY anchor_clm_id
            ORDER BY cpt_claim_count DESC
        ) AS cpt_count_rank,
		ROW_NUMBER() OVER (
            PARTITION BY anchor_clm_id
            ORDER BY cpt_allowed_amt DESC
        ) AS cpt_allowed_rank,
		ROW_NUMBER() OVER (
            PARTITION BY anchor_clm_id
            ORDER BY cpt_coverd_amt DESC
        ) AS cpt_coverd_rank
    FROM CPT_History
)

SELECT cur_clm_uniq_id, 
SUM(CASE WHEN cpt_count_rank=1 THEN cpt_claim_count END) AS billnpi_top1_cpt_clmline_ct_365days,
SUM(CASE WHEN cpt_count_rank<=5 THEN cpt_claim_count END) AS billnpi_top5_cpt_clmline_ct_365days,
SUM(CASE WHEN cpt_allowed_rank=1 THEN cpt_allowed_amt END) AS billnpi_top1_cpt_chrg_amt_365days,
SUM(CASE WHEN cpt_allowed_rank<=5 THEN cpt_allowed_amt END) AS billnpi_top5_cpt_chrg_amt_365days,
SUM(CASE WHEN cpt_coverd_rank=1 THEN cpt_coverd_amt END) AS billnpi_top1_cpt_paid_amt_365days,
SUM(CASE WHEN cpt_coverd_rank<=5 THEN cpt_coverd_amt END) AS billnpi_top5_cpt_paid_amt_365days
into #BillNPI_FEATURES_3
from CPT_Ranked
GROUP BY cur_clm_uniq_id
'''

final_sql_billnpi ='''
SELECT
    a.*,
    b.*,
    c.*
FROM #BillNPI_FEATURES_1 a
LEFT JOIN #BillNPI_FEATURES_2 b
  ON a.cur_clm_uniq_id = b.cur_clm_uniq_id
LEFT JOIN #BillNPI_FEATURES_3 c
  ON a.cur_clm_uniq_id = c.cur_clm_uniq_id
'''

with engine.begin() as conn:   
    conn.exec_driver_sql(sql_build_tables_billnpi)
    df_features_billnpi = pd.read_sql_query(final_sql_billnpi, conn)

In [ ]:
#Calculated Fields

# Avg Allow/Paid Amount Per Patient in 30, 90, 365 days
df_features_billnpi['billnpi_avg_chrg_amt_per_patient_30days']  = df_features_billnpi['billnpi_chrg_amt_30days']  / df_features_billnpi['billnpi_uniq_bene_ct_30days']
df_features_billnpi['billnpi_avg_chrg_amt_per_patient_90days']  = df_features_billnpi['billnpi_chrg_amt_90days']  / df_features_billnpi['billnpi_uniq_bene_ct_90days']
df_features_billnpi['billnpi_avg_chrg_amt_per_patient_365days'] = df_features_billnpi['billnpi_chrg_amt_365days'] / df_features_billnpi['billnpi_uniq_bene_ct_365days']

df_features_billnpi['billnpi_avg_paid_amt_per_patient_30days']  = df_features_billnpi['billnpi_paid_amt_30days']  / df_features_billnpi['billnpi_uniq_bene_ct_30days']
df_features_billnpi['billnpi_avg_paid_amt_per_patient_90days']  = df_features_billnpi['billnpi_paid_amt_90days']  / df_features_billnpi['billnpi_uniq_bene_ct_90days']
df_features_billnpi['billnpi_avg_paid_amt_per_patient_365days'] = df_features_billnpi['billnpi_paid_amt_365days'] / df_features_billnpi['billnpi_uniq_bene_ct_365days']

# Avg Allow/Paid Amount Per Claim in 30, 90, 365 days
df_features_billnpi['billnpi_avg_chrg_amt_per_claim_30days']  = df_features_billnpi['billnpi_chrg_amt_30days']  / df_features_billnpi['billnpi_clmheader_ct_30days']
df_features_billnpi['billnpi_avg_chrg_amt_per_claim_90days']  = df_features_billnpi['billnpi_chrg_amt_90days']  / df_features_billnpi['billnpi_clmheader_ct_90days']
df_features_billnpi['billnpi_avg_chrg_amt_per_claim_365days'] = df_features_billnpi['billnpi_chrg_amt_365days'] / df_features_billnpi['billnpi_clmheader_ct_365days']

df_features_billnpi['billnpi_avg_paid_amt_per_claim_30days']  = df_features_billnpi['billnpi_paid_amt_30days']  / df_features_billnpi['billnpi_clmheader_ct_30days']
df_features_billnpi['billnpi_avg_paid_amt_per_claim_90days']  = df_features_billnpi['billnpi_paid_amt_90days']  / df_features_billnpi['billnpi_clmheader_ct_90days']
df_features_billnpi['billnpi_avg_paid_amt_per_claim_365days'] = df_features_billnpi['billnpi_paid_amt_365days'] / df_features_billnpi['billnpi_clmheader_ct_365days']

# Portion of High Risk CPT
df_features_billnpi['billnpi_risk_cpt_percentage_30days']  = df_features_billnpi['billnpi_risk_cpt_ct_30days']  / df_features_billnpi['billnpi_clmline_ct_30days']
df_features_billnpi['billnpi_risk_cpt_percentage_90days']  = df_features_billnpi['billnpi_risk_cpt_ct_90days']  / df_features_billnpi['billnpi_clmline_ct_90days']
df_features_billnpi['billnpi_risk_cpt_percentage_365days'] = df_features_billnpi['billnpi_risk_cpt_ct_365days'] / df_features_billnpi['billnpi_clmline_ct_365days']

# Top 1/5 CPT count share
df_features_billnpi['billnpi_top1_cpt_percentage_365days'] = df_features_billnpi['billnpi_top1_cpt_clmline_ct_365days'] / df_features_billnpi['billnpi_clmline_ct_365days']
df_features_billnpi['billnpi_top5_cpt_percentage_365days'] = df_features_billnpi['billnpi_top5_cpt_clmline_ct_365days'] / df_features_billnpi['billnpi_clmline_ct_365days']

# Top 1/5 CPT allowed amount share
df_features_billnpi['billnpi_top1_cpt_chrg_amt_percentage_365days'] = df_features_billnpi['billnpi_top1_cpt_chrg_amt_365days'] / df_features_billnpi['billnpi_chrg_amt_365days']
df_features_billnpi['billnpi_top5_cpt_chrg_amt_percentage_365days'] = df_features_billnpi['billnpi_top5_cpt_chrg_amt_365days'] / df_features_billnpi['billnpi_chrg_amt_365days']

# Top 1/5 CPT paid amount share
df_features_billnpi['billnpi_top1_cpt_paid_amt_percentage_365days'] = df_features_billnpi['billnpi_top1_cpt_paid_amt_365days'] / df_features_billnpi['billnpi_paid_amt_365days']
df_features_billnpi['billnpi_top5_cpt_paid_amt_percentage_365days'] = df_features_billnpi['billnpi_top5_cpt_paid_amt_365days'] / df_features_billnpi['billnpi_paid_amt_365days']

# % of new patients in last 365 days
df_features_billnpi['billnpi_new_patients_percentage_365days'] = df_features_billnpi['billnpi_new_bene_ct_365days'] / df_features_billnpi['billnpi_uniq_bene_ct_365days']

# % of 30/90 days claims from last 365 days
df_features_billnpi['billnpi_30days_claim_percentage_to_365days'] = df_features_billnpi['billnpi_clmheader_ct_30days'] / df_features_billnpi['billnpi_clmheader_ct_365days']
df_features_billnpi['billnpi_90days_claim_percentage_to_365days'] = df_features_billnpi['billnpi_clmheader_ct_90days'] / df_features_billnpi['billnpi_clmheader_ct_365days']

In [ ]:
#Export
print(df_features_billnpi.columns)
df_features_billnpi.to_csv(r'U:\Zeming\AAA-Project\DME Fraud Claims Detection\Training Data\features_billnpi.csv')

## **Feature Tables Ordering NPI Level**

In [ ]:
##Ordering NPI Level
sql_build_tables_ordenpi = f'''
DROP TABLE IF EXISTS #DME_ordenpi_DAILY;
SELECT
    ordrg_prvdr_npi_num,
    clm_thru_dt,

    COUNT(*) AS clmline_cnt,
    COUNT(DISTINCT cur_clm_uniq_id) AS dist_claim_cnt,
    COUNT(DISTINCT Beneficiary_ID) AS dist_bene_cnt,
    COUNT(DISTINCT clm_line_hcpcs_cd) AS dist_cpt_cnt,

    SUM(clm_line_alowd_chrg_amt) AS allowed_amt,
    SUM(clm_line_cvrd_pd_amt) AS paid_amt,

    SUM(CASE WHEN c.CPT IS NOT NULL THEN 1 ELSE 0 END) AS risk_cpt_cnt
INTO #DME_ordenpi_DAILY
FROM temp_zeming.dbo.dme d
LEFT JOIN (
    SELECT DISTINCT CPT
    FROM temp_zeming.dbo.DMEAutomation_CPT
) c
  ON d.clm_line_hcpcs_cd = c.CPT
GROUP BY
    ordrg_prvdr_npi_num,
    clm_thru_dt;

DROP TABLE IF EXISTS #ordenpi_FEATURES_1;
SELECT
    a.cur_clm_uniq_id,

    -- claim / line
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.clmline_cnt END) AS ordenpi_clmline_ct_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.clmline_cnt END) AS ordenpi_clmline_ct_90days,
    SUM(b.clmline_cnt) AS ordenpi_clmline_ct_365days,

    -- distinct claim
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.dist_claim_cnt END) AS ordenpi_clmheader_ct_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.dist_claim_cnt END) AS ordenpi_clmheader_ct_90days,
    SUM(b.dist_claim_cnt) AS ordenpi_clmheader_ct_365days,

    -- distinct bene
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.dist_bene_cnt END) AS ordenpi_uniq_bene_ct_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.dist_bene_cnt END) AS ordenpi_uniq_bene_ct_90days,
    SUM(b.dist_bene_cnt) AS ordenpi_uniq_bene_ct_365days,

    -- amounts
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.allowed_amt END) AS ordenpi_chrg_amt_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.allowed_amt END) AS ordenpi_chrg_amt_90days,
    SUM(b.allowed_amt) AS ordenpi_chrg_amt_365days,

    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.paid_amt END) AS ordenpi_paid_amt_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.paid_amt END) AS ordenpi_paid_amt_90days,
    SUM(b.paid_amt) AS ordenpi_paid_amt_365days,

    -- risk CPT
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-30,a.clm_thru_dt) THEN b.risk_cpt_cnt END) AS ordenpi_risk_cpt_ct_30days,
    SUM(CASE WHEN b.clm_thru_dt >= DATEADD(day,-90,a.clm_thru_dt) THEN b.risk_cpt_cnt END) AS ordenpi_risk_cpt_ct_90days,
    SUM(b.risk_cpt_cnt) AS ordenpi_risk_cpt_ct_365days

INTO #ordenpi_FEATURES_1
FROM temp_zeming.dbo.dme_header a
JOIN #DME_ordenpi_DAILY b
  ON a.ordrg_prvdr_npi_num = b.ordrg_prvdr_npi_num
 AND b.clm_thru_dt BETWEEN DATEADD(day,-365,a.clm_thru_dt)
                       AND a.clm_thru_dt
WHERE a.clm_thru_dt >= '2021-01-01'
GROUP BY
    a.cur_clm_uniq_id;

--New Patient Counts
;WITH Claim_With_New_Flag AS (
    SELECT
        a.cur_clm_uniq_id,
        a.Beneficiary_ID,
        a.ordrg_prvdr_npi_num,
        a.clm_thru_dt,

        CASE 
            WHEN NOT EXISTS (
                SELECT 1
                FROM temp_zeming.dbo.dme_header x
                WHERE x.Beneficiary_ID = a.Beneficiary_ID
                  AND x.ordrg_prvdr_npi_num = a.ordrg_prvdr_npi_num
                  AND x.clm_thru_dt <= a.clm_thru_dt
                  AND x.clm_thru_dt >= DATEADD(year, -2, a.clm_thru_dt)
				  AND x.cur_clm_uniq_id != a.cur_clm_uniq_id
            )
            THEN 1
            ELSE 0
        END AS is_new_patient_provider
    FROM temp_zeming.dbo.dme_header a
)

SELECT
    a.cur_clm_uniq_id,

    COUNT(DISTINCT CASE 
        WHEN b.is_new_patient_provider = 1
         AND b.clm_thru_dt >= DATEADD(day, -30, a.clm_thru_dt)
        THEN b.Beneficiary_ID
    END) AS ordenpi_new_bene_ct_30days,

    COUNT(DISTINCT CASE 
        WHEN b.is_new_patient_provider = 1
         AND b.clm_thru_dt >= DATEADD(day, -90, a.clm_thru_dt)
        THEN b.Beneficiary_ID
    END) AS ordenpi_new_bene_ct_90days,

    COUNT(DISTINCT CASE 
        WHEN b.is_new_patient_provider = 1
         AND b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.Beneficiary_ID
    END) AS ordenpi_new_bene_ct_365days
into #ordenpi_FEATURES_2
FROM Claim_With_New_Flag a
LEFT JOIN Claim_With_New_Flag b
  ON a.ordrg_prvdr_npi_num = b.ordrg_prvdr_npi_num
 AND b.clm_thru_dt <= a.clm_thru_dt
 AND b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
 AND b.cur_clm_uniq_id != a.cur_clm_uniq_id
WHERE a.clm_thru_dt >= '2021-01-01'
GROUP BY
    a.cur_clm_uniq_id;


--Top1, Top5 CPT count, allowed amount, paid amount in last 365 days for this provider
WITH CPT_History AS (
    SELECT
        a.cur_clm_uniq_id AS anchor_clm_id,
        a.ordrg_prvdr_npi_num,
        b.clm_line_hcpcs_cd,
        COUNT(b.cur_clm_uniq_id) AS cpt_claim_count,
		SUM(B.clm_line_alowd_chrg_amt) AS cpt_allowed_amt,
		SUM(B. clm_line_cvrd_pd_amt) as cpt_coverd_amt
    FROM temp_zeming.dbo.dme_header a
    JOIN temp_zeming.dbo.dme b
      ON a.ordrg_prvdr_npi_num = b.ordrg_prvdr_npi_num
     AND b.clm_thru_dt <= a.clm_thru_dt
     AND b.clm_thru_dt >= DATEADD(year, -1, a.clm_thru_dt)  -- lookback
    GROUP BY
        a.cur_clm_uniq_id,
        a.ordrg_prvdr_npi_num,
        b.clm_line_hcpcs_cd
)

, CPT_Ranked AS (
    SELECT
        anchor_clm_id AS cur_clm_uniq_id,
        ordrg_prvdr_npi_num,
        clm_line_hcpcs_cd,
        cpt_claim_count,
		cpt_allowed_amt,
		cpt_coverd_amt,
        ROW_NUMBER() OVER (
            PARTITION BY anchor_clm_id
            ORDER BY cpt_claim_count DESC
        ) AS cpt_count_rank,
		ROW_NUMBER() OVER (
            PARTITION BY anchor_clm_id
            ORDER BY cpt_allowed_amt DESC
        ) AS cpt_allowed_rank,
		ROW_NUMBER() OVER (
            PARTITION BY anchor_clm_id
            ORDER BY cpt_coverd_amt DESC
        ) AS cpt_coverd_rank
    FROM CPT_History
)

SELECT cur_clm_uniq_id, 
SUM(CASE WHEN cpt_count_rank=1 THEN cpt_claim_count END) AS ordenpi_top1_cpt_clmline_ct_365days,
SUM(CASE WHEN cpt_count_rank<=5 THEN cpt_claim_count END) AS ordenpi_top5_cpt_clmline_ct_365days,
SUM(CASE WHEN cpt_allowed_rank=1 THEN cpt_allowed_amt END) AS ordenpi_top1_cpt_chrg_amt_365days,
SUM(CASE WHEN cpt_allowed_rank<=5 THEN cpt_allowed_amt END) AS ordenpi_top5_cpt_chrg_amt_365days,
SUM(CASE WHEN cpt_coverd_rank=1 THEN cpt_coverd_amt END) AS ordenpi_top1_cpt_paid_amt_365days,
SUM(CASE WHEN cpt_coverd_rank<=5 THEN cpt_coverd_amt END) AS ordenpi_top5_cpt_paid_amt_365days
into #ordenpi_FEATURES_3
from CPT_Ranked
GROUP BY cur_clm_uniq_id
'''

final_sql_ordenpi ='''
SELECT
    a.*,
    b.*,
    c.*
FROM #ordenpi_FEATURES_1 a
LEFT JOIN #ordenpi_FEATURES_2 b
  ON a.cur_clm_uniq_id = b.cur_clm_uniq_id
LEFT JOIN #ordenpi_FEATURES_3 c
  ON a.cur_clm_uniq_id = c.cur_clm_uniq_id
'''

with engine.begin() as conn:   
    conn.exec_driver_sql(sql_build_tables_ordenpi)
    df_features_ordenpi = pd.read_sql_query(final_sql_ordenpi, conn)

In [ ]:
#Calculated Fields

# Avg Allow/Paid Amount Per Patient in 30, 90, 365 days
df_features_ordenpi['ordenpi_avg_chrg_amt_per_patient_30days']  = df_features_ordenpi['ordenpi_chrg_amt_30days']  / df_features_ordenpi['ordenpi_uniq_bene_ct_30days']
df_features_ordenpi['ordenpi_avg_chrg_amt_per_patient_90days']  = df_features_ordenpi['ordenpi_chrg_amt_90days']  / df_features_ordenpi['ordenpi_uniq_bene_ct_90days']
df_features_ordenpi['ordenpi_avg_chrg_amt_per_patient_365days'] = df_features_ordenpi['ordenpi_chrg_amt_365days'] / df_features_ordenpi['ordenpi_uniq_bene_ct_365days']

df_features_ordenpi['ordenpi_avg_paid_amt_per_patient_30days']  = df_features_ordenpi['ordenpi_paid_amt_30days']  / df_features_ordenpi['ordenpi_uniq_bene_ct_30days']
df_features_ordenpi['ordenpi_avg_paid_amt_per_patient_90days']  = df_features_ordenpi['ordenpi_paid_amt_90days']  / df_features_ordenpi['ordenpi_uniq_bene_ct_90days']
df_features_ordenpi['ordenpi_avg_paid_amt_per_patient_365days'] = df_features_ordenpi['ordenpi_paid_amt_365days'] / df_features_ordenpi['ordenpi_uniq_bene_ct_365days']

# Avg Allow/Paid Amount Per Claim in 30, 90, 365 days
df_features_ordenpi['ordenpi_avg_chrg_amt_per_claim_30days']  = df_features_ordenpi['ordenpi_chrg_amt_30days']  / df_features_ordenpi['ordenpi_clmheader_ct_30days']
df_features_ordenpi['ordenpi_avg_chrg_amt_per_claim_90days']  = df_features_ordenpi['ordenpi_chrg_amt_90days']  / df_features_ordenpi['ordenpi_clmheader_ct_90days']
df_features_ordenpi['ordenpi_avg_chrg_amt_per_claim_365days'] = df_features_ordenpi['ordenpi_chrg_amt_365days'] / df_features_ordenpi['ordenpi_clmheader_ct_365days']

df_features_ordenpi['ordenpi_avg_paid_amt_per_claim_30days']  = df_features_ordenpi['ordenpi_paid_amt_30days']  / df_features_ordenpi['ordenpi_clmheader_ct_30days']
df_features_ordenpi['ordenpi_avg_paid_amt_per_claim_90days']  = df_features_ordenpi['ordenpi_paid_amt_90days']  / df_features_ordenpi['ordenpi_clmheader_ct_90days']
df_features_ordenpi['ordenpi_avg_paid_amt_per_claim_365days'] = df_features_ordenpi['ordenpi_paid_amt_365days'] / df_features_ordenpi['ordenpi_clmheader_ct_365days']

# Portion of High Risk CPT
df_features_ordenpi['ordenpi_risk_cpt_percentage_30days']  = df_features_ordenpi['ordenpi_risk_cpt_ct_30days']  / df_features_ordenpi['ordenpi_clmline_ct_30days']
df_features_ordenpi['ordenpi_risk_cpt_percentage_90days']  = df_features_ordenpi['ordenpi_risk_cpt_ct_90days']  / df_features_ordenpi['ordenpi_clmline_ct_90days']
df_features_ordenpi['ordenpi_risk_cpt_percentage_365days'] = df_features_ordenpi['ordenpi_risk_cpt_ct_365days'] / df_features_ordenpi['ordenpi_clmline_ct_365days']

# Top 1/5 CPT count share
df_features_ordenpi['ordenpi_top1_cpt_percentage_365days'] = df_features_ordenpi['ordenpi_top1_cpt_clmline_ct_365days'] / df_features_ordenpi['ordenpi_clmline_ct_365days']
df_features_ordenpi['ordenpi_top5_cpt_percentage_365days'] = df_features_ordenpi['ordenpi_top5_cpt_clmline_ct_365days'] / df_features_ordenpi['ordenpi_clmline_ct_365days']

# Top 1/5 CPT allowed amount share
df_features_ordenpi['ordenpi_top1_cpt_chrg_amt_percentage_365days'] = df_features_ordenpi['ordenpi_top1_cpt_chrg_amt_365days'] / df_features_ordenpi['ordenpi_chrg_amt_365days']
df_features_ordenpi['ordenpi_top5_cpt_chrg_amt_percentage_365days'] = df_features_ordenpi['ordenpi_top5_cpt_chrg_amt_365days'] / df_features_ordenpi['ordenpi_chrg_amt_365days']

# Top 1/5 CPT paid amount share
df_features_ordenpi['ordenpi_top1_cpt_paid_amt_percentage_365days'] = df_features_ordenpi['ordenpi_top1_cpt_paid_amt_365days'] / df_features_ordenpi['ordenpi_paid_amt_365days']
df_features_ordenpi['ordenpi_top5_cpt_paid_amt_percentage_365days'] = df_features_ordenpi['ordenpi_top5_cpt_paid_amt_365days'] / df_features_ordenpi['ordenpi_paid_amt_365days']

# % of new patients in last 365 days
df_features_ordenpi['ordenpi_new_patients_percentage_365days'] = df_features_ordenpi['ordenpi_new_bene_ct_365days'] / df_features_ordenpi['ordenpi_uniq_bene_ct_365days']

# % of 30/90 days claims from last 365 days
df_features_ordenpi['ordenpi_30days_claim_percentage_to_365days'] = df_features_ordenpi['ordenpi_clmheader_ct_30days'] / df_features_ordenpi['ordenpi_clmheader_ct_365days']
df_features_ordenpi['ordenpi_90days_claim_percentage_to_365days'] = df_features_ordenpi['ordenpi_clmheader_ct_90days'] / df_features_ordenpi['ordenpi_clmheader_ct_365days']

In [ ]:
#Export
print(df_features_ordenpi.columns)
df_features_ordenpi.to_csv(r'U:\Zeming\AAA-Project\DME Fraud Claims Detection\Training Data\features_ordenpi.csv')

## **Feature Tables Billing NPI Bene Level**

In [ ]:
##NPI-Bene Level
sql_build_tables_billnpibene = '''
--Look Back 30, 90, 365days
DROP TABLE IF EXISTS #billnpibene_FEATURES_1
SELECT
    a.cur_clm_uniq_id,

--Cliam Line Counts
    COUNT( CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS billnpibene_clmline_ct_30days,

    COUNT( CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS billnpibene_clmline_ct_90days,

    COUNT( CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS billnpibene_clmline_ct_365days,

--Distinct Claim Counts
    COUNT( DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS billnpibene_clmheader_ct_30days,

    COUNT( DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS billnpibene_clmheader_ct_90days,

    COUNT( DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS billnpibene_clmheader_ct_365days,

	--distinct hcpcs
	COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.clm_line_hcpcs_cd
    END) AS billnpibene_uniq_cpt_ct_30days,

    COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.clm_line_hcpcs_cd
    END) AS billnpibene_uniq_cpt_ct_90days,

    COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.clm_line_hcpcs_cd
    END) AS billnpibene_uniq_cpt_ct_365days,

	--total allowed amount
	SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.clm_line_alowd_chrg_amt
    END) AS billnpibene_chrg_amt_30days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.clm_line_alowd_chrg_amt
    END) AS billnpibene_chrg_amt_90days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.clm_line_alowd_chrg_amt
    END) AS billnpibene_chrg_amt_365days,

	--total paid amount
	SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.clm_line_cvrd_pd_amt
    END) AS billnpibene_paid_amt_30days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.clm_line_cvrd_pd_amt
    END) AS billnpibene_paid_amt_90days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.clm_line_cvrd_pd_amt
    END) AS billnpibene_paid_amt_365days

	
INTO #BillNPIbene_FEATURES_1
FROM temp_zeming.dbo.dme_header a
JOIN temp_zeming.dbo.dme b
  ON a.pay_to_prvdr_npi_num = b.pay_to_prvdr_npi_num
  AND A.Beneficiary_ID =B.Beneficiary_ID
  AND b.clm_thru_dt <= a.clm_thru_dt --claims before the anchor date
  AND b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)   -- maxmium 365 days look back
WHERE a.clm_thru_dt >= '2021-01-01'
GROUP BY
    a.cur_clm_uniq_id
'''

final_sql_billnpibene = """
select * from #BillNPIbene_FEATURES_1;
"""

with engine.begin() as conn:
    conn.exec_driver_sql(sql_build_tables_billnpibene)
    df_features_billnpibene = pd.read_sql_query(final_sql_billnpibene, conn)

In [ ]:
df_features_billnpibene.columns
df_features_billnpibene.to_csv(r'U:\Zeming\AAA-Project\DME Fraud Claims Detection\Training Data\features_billnpibene.csv')

## **Feature Tables Ordering NPI Bene Level**

In [ ]:
##NPI-Bene Level
sql_build_tables_ordenpibene = '''
--Look Back 30, 90, 365days
DROP TABLE IF EXISTS #ordenpibene_FEATURES_1
SELECT
    a.cur_clm_uniq_id,

--Cliam Line Counts
    COUNT( CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS ordenpibene_clmline_ct_30days,

    COUNT( CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS ordenpibene_clmline_ct_90days,

    COUNT( CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS ordenpibene_clmline_ct_365days,

--Distinct Claim Counts
    COUNT( DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS ordenpibene_clmheader_ct_30days,

    COUNT( DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS ordenpibene_clmheader_ct_90days,

    COUNT( DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.cur_clm_uniq_id
    END) AS ordenpibene_clmheader_ct_365days,

	--distinct hcpcs
	COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.clm_line_hcpcs_cd
    END) AS ordenpibene_uniq_cpt_ct_30days,

    COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.clm_line_hcpcs_cd
    END) AS ordenpibene_uniq_cpt_ct_90days,

    COUNT(DISTINCT CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.clm_line_hcpcs_cd
    END) AS ordenpibene_uniq_cpt_ct_365days,

	--total allowed amount
	SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.clm_line_alowd_chrg_amt
    END) AS ordenpibene_chrg_amt_30days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.clm_line_alowd_chrg_amt
    END) AS ordenpibene_chrg_amt_90days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.clm_line_alowd_chrg_amt
    END) AS ordenpibene_chrg_amt_365days,

	--total paid amount
	SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -30,  a.clm_thru_dt)
        THEN b.clm_line_cvrd_pd_amt
    END) AS ordenpibene_paid_amt_30days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -90,  a.clm_thru_dt)
        THEN b.clm_line_cvrd_pd_amt
    END) AS ordenpibene_paid_amt_90days,

    SUM(CASE 
        WHEN b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)
        THEN b.clm_line_cvrd_pd_amt
    END) AS ordenpibene_paid_amt_365days

	
INTO #ordenpibene_FEATURES_1
FROM temp_zeming.dbo.dme_header a
JOIN temp_zeming.dbo.dme b
  ON a.ordrg_prvdr_npi_num = b.ordrg_prvdr_npi_num
  AND A.Beneficiary_ID =B.Beneficiary_ID
  AND b.clm_thru_dt <= a.clm_thru_dt --claims before the anchor date
  AND b.clm_thru_dt >= DATEADD(day, -365, a.clm_thru_dt)   -- maxmium 365 days look back
WHERE a.clm_thru_dt >= '2021-01-01'
GROUP BY
    a.cur_clm_uniq_id
'''

final_sql_ordenpibene = """
select * from #ordenpibene_FEATURES_1;
"""

with engine.begin() as conn:
    conn.exec_driver_sql(sql_build_tables_ordenpibene)
    df_features_ordenpibene = pd.read_sql_query(final_sql_ordenpibene, conn)

In [ ]:
df_features_ordenpibene.columns
df_features_ordenpibene.to_csv(r'U:\Zeming\AAA-Project\DME Fraud Claims Detection\Training Data\features_ordenpibene.csv')

## **Export Final Trannning Data**

In [ ]:
print(len(df_features_claim))
print(len(df_features_bene))
print(len(df_features_billnpi))
print(len(df_features_billnpibene))
print(len(df_features_ordenpi))
print(len(df_features_ordenpibene))

In [ ]:
#drop duplicate columns
df_features_bene = df_features_bene.loc[:,~df_features_bene.columns.duplicated()]
df_features_billnpi = df_features_billnpi.loc[:,~df_features_billnpi.columns.duplicated()]
df_features_billnpibene = df_features_billnpibene.loc[:,~df_features_billnpibene.columns.duplicated()]
df_features_ordenpi = df_features_ordenpi.loc[:,~df_features_ordenpi.columns.duplicated()]
df_features_ordenpibene = df_features_ordenpibene.loc[:,~df_features_ordenpibene.columns.duplicated()]

#merge
df_training = (
    df_features_claim
        .merge(df_features_bene, on='cur_clm_uniq_id', how='left')
        .merge(df_features_billnpi, on='cur_clm_uniq_id', how='left')
        .merge(df_features_billnpibene, on='cur_clm_uniq_id', how='left')
        .merge(df_features_ordenpi, on='cur_clm_uniq_id', how='left')
        .merge(df_features_ordenpibene, on='cur_clm_uniq_id', how='left')
)

In [ ]:
#Final Calculated Fields

# % of billnpi claim line for this patient in 30, 90, 365 days
# % of billnpi distinct claim header for this patient in 30, 90, 365 days
# % of billnpi paid amount for this patient in 30, 90, 365 days
# % of billnpi allowed amount for this patient in 30, 90, 365 days
df_training['billnpibene_%claimline_on_this_patient_365days'] = df_training['billnpibene_clmline_ct_365days']/df_training['billnpi_clmline_ct_365days']
df_training['billnpibene_%claimheader_on_this_patient_365days'] = df_training['billnpibene_clmheader_ct_365days']/df_training['billnpi_clmheader_ct_365days']
df_training['billnpibene_%paid_on_this_patient_365days'] = df_training['billnpibene_paid_amt_365days']/df_training['billnpi_paid_amt_365days']
df_training['billnpibene_%chrg_on_this_patient_365days'] = df_training['billnpibene_chrg_amt_365days']/df_training['billnpi_chrg_amt_365days']

df_training['ordenpibene_%claimline_on_this_patient_365days'] = df_training['ordenpibene_clmline_ct_365days']/df_training['ordenpi_clmline_ct_365days']
df_training['ordenpibene_%claimheader_on_this_patient_365days'] = df_training['ordenpibene_clmheader_ct_365days']/df_training['ordenpi_clmheader_ct_365days']
df_training['ordenpibene_%paid_on_this_patient_365days'] = df_training['ordenpibene_paid_amt_365days']/df_training['ordenpi_paid_amt_365days']
df_training['ordenpibene_%chrg_on_this_patient_365days'] = df_training['ordenpibene_chrg_amt_365days']/df_training['ordenpi_chrg_amt_365days']

In [ ]:
df_training.to_csv(r'U:\Zeming\AAA-Project\DME Fraud Claims Detection\Training Data\raw_training_data.csv',index=False)